## Outlier Analysis & Handling

In [1]:
import pandas as pd

listings = pd.read_pickle("..\\processed\\CRMLSListings_Engineered.pkl")
sold = pd.read_pickle("..\\processed\\CRMLSSold_Engineered.pkl")

Form outlier ranges and flag outliers

In [9]:
metrics = {"ClosePrice": "ClosePriceOutlier", "LivingArea": "LivingAreaOutlier",
           "Price Per Sq Ft": "PricePerSqFtOutlier", "Price Ratio": "PriceRatioOutlier",
           "Days on Market": "DaysOnMarketOutlier", "Close to Original List Ratio": "CloseToOriginalListPriceOutlier"}

quantiles = [0.25, 0.5, 0.75]
summaries = {}
for name, df in [("Sold", sold), ("Listings", listings)]:
    summaries[name] = df[list(metrics)].describe(percentiles=quantiles).T

In [3]:
for metric, outlier_metric in metrics.items():
    Q1 = summaries["Listings"].loc[metric, "25%"]
    Q3 = summaries["Listings"].loc[metric, "75%"]
    IQR = Q3 - Q1
    listings[outlier_metric] = (listings[metric] < Q1 - 1.5 * IQR) | (listings[metric] > Q3 + 1.5 * IQR)
    sold[outlier_metric] = (sold[metric] < Q1 - 1.5 * IQR) | (sold[metric] > Q3 + 1.5 * IQR)

for name, df in [("Sold", sold), ("Listings", listings)]:
    df["AnyOutlier"] = df[list(metrics.values())].any(axis=1)
    print(f"\n{name} outliers:")
    any_outliers = df["AnyOutlier"].sum()
    print(
        f"AnyOutlier: {any_outliers:,} / {len(df):,} "
        f"({any_outliers / len(df) * 100:.2f}%)"
    )

    for outlier_metric in metrics.values():
        matched = df[outlier_metric].sum()
        print(
            f"{outlier_metric}: {matched:,} / {len(df):,} "
            f"({matched / len(df) * 100:.2f}%)"
        )


Sold outliers:
AnyOutlier: 184,813 / 480,314 (38.48%)
ClosePriceOutlier: 33,042 / 480,314 (6.88%)
LivingAreaOutlier: 17,137 / 480,314 (3.57%)
PricePerSqFtOutlier: 18,841 / 480,314 (3.92%)
PriceRatioOutlier: 52,520 / 480,314 (10.93%)
DaysOnMarketOutlier: 112,478 / 480,314 (23.42%)
CloseToOriginalListPriceOutlier: 56,496 / 480,314 (11.76%)

Listings outliers:
AnyOutlier: 123,206 / 644,002 (19.13%)
ClosePriceOutlier: 13,886 / 644,002 (2.16%)
LivingAreaOutlier: 31,612 / 644,002 (4.91%)
PricePerSqFtOutlier: 8,741 / 644,002 (1.36%)
PriceRatioOutlier: 22,367 / 644,002 (3.47%)
DaysOnMarketOutlier: 66,357 / 644,002 (10.30%)
CloseToOriginalListPriceOutlier: 19,849 / 644,002 (3.08%)


Compare summary statistics with and without outliers

In [8]:
formatters = {
    "count": "{:,.0f}", "mean": "{:,.2f}", "std": "{:,.2f}",
    "min": "{:,.2f}", "25%": "{:,.2f}", "50%": "{:,.2f}",
    "75%": "{:,.2f}", "max": "{:,.2f}",
}

for name, df in [("Sold", sold), ("Listings", listings)]:
    print(f"\n{name} metric statistics before and after outlier filtering:")
    for metric, outlier_metric in metrics.items():
        filtered = df.loc[~df[outlier_metric], [metric]]
        after = filtered.describe(percentiles=quantiles).T
        comparison = pd.DataFrame(
            {"Before": summaries[name].loc[metric], "After": after.loc[metric]}
        ).T
        print(f"\n{metric} (excluding {metric} outliers, {len(filtered):,} rows):")
        display(comparison.style.format(formatters, na_rep="N/A"))

    filtered_all = df.loc[~df["AnyOutlier"], list(metrics)]
    after_all = filtered_all.describe(percentiles=quantiles).T
    comparison_all = pd.concat(
        {"Before": summaries[name], "After": after_all}, names=["Status"]
    ).rename_axis(index=["Status", "Metric"]).reset_index()
    print(f"\nAll metrics (excluding all outliers, {len(filtered_all):,} rows):")
    display(comparison_all.style.format(formatters, na_rep="N/A"))


Sold metric statistics before and after outlier filtering:

ClosePrice (excluding ClosePrice outliers, 447,272 rows):


,count,mean,std,min,25%,50%,75%,max
Before,"480,312","1,188,811.48","5,864,009.10",1.15,"575,000.00","825,000.00","1,300,000.00","989,500,000.00"
After,"447,270","909,654.92","483,884.59",1.15,"559,000.00","785,000.00","1,170,000.00","2,475,650.00"



LivingArea (excluding LivingArea outliers, 463,177 rows):


,count,mean,std,min,25%,50%,75%,max
Before,"480,045","1,903.73","24,585.05",1.00,"1,250.00","1,648.00","2,227.00","17,021,321.00"
After,"462,908","1,746.34",688.95,1.00,"1,236.00","1,616.00","2,141.00","3,878.00"



Price Per Sq Ft (excluding Price Per Sq Ft outliers, 461,473 rows):


,count,mean,std,min,25%,50%,75%,max
Before,"480,043",642.52,"4,874.49",0.00,367.46,536.42,731.50,"1,164,066.85"
After,"461,202",554.14,246.05,0.00,360.26,523.53,701.79,"1,305.91"


... (18 outputs left out)


Close to Original List Ratio (excluding Close to Original List Ratio outliers, 624,153 rows):


,count,mean,std,min,25%,50%,75%,max
Before,"192,135",inf,N/A,0.00,0.97,1.00,1.03,inf
After,"172,286",0.99,0.04,0.88,0.97,1.00,1.02,1.11



All metrics (excluding all outliers, 520,796 rows):


,Status,Metric,count,mean,std,min,25%,50%,75%,max
0,Before,ClosePrice,"193,065","1,195,945.18","3,954,604.10",525.00,"599,500.00","850,000.00","1,350,000.00","820,000,000.00"
1,Before,LivingArea,"643,381","1,974.91","21,421.58",1.00,"1,248.00","1,671.00","2,300.00","17,021,321.00"
2,Before,Price Per Sq Ft,"192,968",650.00,"3,441.85",0.37,391.08,558.87,757.03,"684,474.12"
3,Before,Price Ratio,"193,065",2.10,455.68,0.00,0.98,1.00,1.02,"200,200.00"
4,Before,Days on Market,"644,002",21.16,28.81,0.00,6.00,11.00,24.00,"1,155.00"
5,Before,Close to Original List Ratio,"192,135",inf,N/A,0.00,0.97,1.00,1.03,inf
6,After,ClosePrice,"132,475","902,644.38","453,089.71","5,000.00","577,950.00","795,000.00","1,140,000.00","2,475,650.00"
7,After,LivingArea,"520,292","1,745.90",701.10,1.00,"1,225.00","1,616.00","2,151.00","3,878.00"
8,After,Price Per Sq Ft,"132,440",554.40,226.56,1.74,380.12,531.91,692.04,"1,305.79"
9,After,Price Ratio,"132,475",1.00,0.03,0.92,0.99,1.00,1.02,1.09


### Save flagged dataset

In [5]:
sold.to_pickle("..\\processed\\CRMLSSold_OutlierFlagged.pkl")
listings.to_pickle("..\\processed\\CRMLSListings_OutlierFlagged.pkl")

sold.to_csv("..\\processed\\CRMLSSold_OutlierFlagged.csv")
listings.to_csv("..\\processed\\CRMLSListings_OutlierFlagged.csv")